# Part 1: Build knowledge bases with agentic retrieval

In Foundry IQ (Azure AI Search), a **knowledge base** is a top-level resource that connects a chat completion model to one or more knowledge sources. It defines which data sources to query, which model to use for reasoning, and how query execution should be optimized.

In this notebook, you'll explore two types of knowledge sources:

1. **File Knowledge Source** — Upload raw files directly; the service handles chunking, embedding, and indexing automatically
2. **Indexed Knowledge Sources** — Connect to pre-built search indexes for production workloads with full control over schema and processing

## Pre-configured environment

This lab uses the following Azure resources, already set up for you:

* Azure AI Search
  * `hrdocs` index: HR policies, handbook, company info
  * `healthdocs` index: health benefits, insurance, coverage
  * Semantic ranking enabled, pre-computed embeddings
* Azure OpenAI from Microsoft Foundry Models
  * `gpt-4.1`: Chat completion for reasoning and synthesis
  * `text-embedding-3-large`: Embedding model for vector search
* Microsoft Fabric

## Step 1: Setup environment variables

Load the configuration for your Azure resources. The `.env` file in the project folder has everything you need: Azure AI Search endpoints, Azure OpenAI credentials, and model configuration.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
AZURE_OPENAI_EMBEDDING_MODEL = "text-embedding-3-large"

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")


---

## A: File Knowledge Source

A **File Knowledge Source** lets you upload raw files directly — the service handles chunking, embedding, and indexing automatically. This is the fastest way to get documents into a knowledge base.

## Step 2: Create a File Knowledge Source

Configure a file knowledge source with an embedding model for automatic ingestion. The `ingestion_parameters` specify:
- **Embedding model**: Generates vector embeddings for the uploaded content
- **Content extraction mode**: How to extract text from documents

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    FileKnowledgeSource,
    FileKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeSourceAzureOpenAIVectorizer,
    KnowledgeSourceIngestionParameters,
)

FILE_KNOWLEDGE_SOURCE_NAME = "file-docs-knowledge-source"

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

file_ks = FileKnowledgeSource(
    name=FILE_KNOWLEDGE_SOURCE_NAME,
    description="Uploaded HR and company documents for Zava",
    file_parameters=FileKnowledgeSourceParameters(
        ingestion_parameters=KnowledgeSourceIngestionParameters(
            embedding_model=KnowledgeSourceAzureOpenAIVectorizer(
                azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                    resource_url=AZURE_OPENAI_ENDPOINT,
                    deployment_name=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
                    model_name=AZURE_OPENAI_EMBEDDING_MODEL,
                    api_key=AZURE_OPENAI_KEY,
                )
            ),
            content_extraction_mode="minimal",
        )
    ),
)

index_client.create_or_update_knowledge_source(file_ks)
print(f"File knowledge source '{FILE_KNOWLEDGE_SOURCE_NAME}' created or updated successfully.")

## Step 3: Upload files to the knowledge source

Upload a document directly to the file knowledge source. The service will automatically extract text, chunk it into searchable segments, and generate vector embeddings.

In [ ]:
from pathlib import Path

data_dir = Path("..") / "data" / "ai-search-data"
files_to_upload = [
    data_dir / "filesource" / "MSFT_cloud_architecture_zava.pdf"
]

uploaded_files = []
for file_path in files_to_upload:
    with open(file_path, "rb") as f:
        file_content = f.read()
    uploaded = index_client.upload_knowledge_source_file(
        FILE_KNOWLEDGE_SOURCE_NAME, file_content
    )
    uploaded_files.append(uploaded)
    print(f"Uploaded: {file_path.name} (file_id: {uploaded.file_id}, {uploaded.file_size_bytes} bytes)")

print(f"\nTotal files uploaded: {len(uploaded_files)}")

## Step 4: Create a knowledge base for the file source

Create a knowledge base that references the file knowledge source. With `output_mode=ANSWER_SYNTHESIS`, the knowledge base uses the chat model to generate natural language answers from the retrieved document chunks.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalOutputMode,
)

FILE_KNOWLEDGE_BASE_NAME = "file-docs-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=FILE_KNOWLEDGE_BASE_NAME,
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[
        KnowledgeSourceReference(name=FILE_KNOWLEDGE_SOURCE_NAME),
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{FILE_KNOWLEDGE_BASE_NAME}' created or updated successfully.")

## Step 5: Query the file knowledge base

Ask a question about the uploaded document. The knowledge base will search across the file content and synthesize a natural language answer with citations.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FileKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
)
from IPython.display import Markdown, display

file_kb_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=FILE_KNOWLEDGE_BASE_NAME, credential=search_credential
)

file_ks_params = FileKnowledgeSourceParams(
    knowledge_source_name=FILE_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)

req = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[
                KnowledgeBaseMessageTextContent(
                    text="What are the levels of Zava data sensitivity classification?"
                )
            ],
        )
    ],
    knowledge_source_params=[file_ks_params],
    include_activity=True,
)

result = file_kb_client.retrieve(retrieval_request=req)
display(Markdown(result.response[0].content[0].text))

activity_by_id = {activity.id: getattr(activity, "knowledge_source_name", None) for activity in (result.activity or [])}
references_json = []

for reference in result.references or []:
    reference_json = reference.as_dict()
    reference_json["knowledgeSourceName"] = activity_by_id.get(reference.activity_source, "unknown")
    references_json.append(reference_json)

print(json.dumps(references_json, indent=2))

---

## B: Indexed Knowledge Sources

**Indexed Knowledge Sources** connect to pre-built search indexes, giving you full control over schema, chunking, and embedding. This is ideal for production workloads with large datasets.

For this section, two search indexes were pre-populated with ingested document chunks as part of the lab setup.

## Step 6: Verify search indexes

Run the cell below to confirm each index has the expected number of document chunks:

* `hrdocs`: 50 document chunks about HR policies
* `healthdocs`: 334 document chunks about health benefits and insurance

In [ ]:
from azure.search.documents import SearchClient

for index_name in [HRDOCS_INDEX, HEALTHDOCS_INDEX]:
    client = SearchClient(AZURE_SEARCH_SERVICE_ENDPOINT, index_name, search_credential)
    count = client.get_document_count()
    print(f"{index_name}: {count} documents")


## Step 7: Create two indexed knowledge sources

A **knowledge source** connects your knowledge base to actual data. You'll create two knowledge sources pointing to each index.

* `healthdocs-knowledge-source`: Points to the `healthdocs` index
* `hrdocs-knowledge-source`: Points to the `hrdocs` index

Both sources set these fields as the searchable fields:

* `snippet`: The actual text content
* `blob_path`: Where the original document lives (used for citation links)

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [HR_KNOWLEDGE_SOURCE_NAME, HEALTH_KNOWLEDGE_SOURCE_NAME]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "HR documents from the restored hrdocs index"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Health benefits documents from the restored healthdocs index"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)
    print(f"Knowledge source '{knowledge_source.name}' created or updated successfully.")


## Step 8: Create the multi-source knowledge base

A knowledge base is the orchestration layer that combines:

1. Your data sources (the knowledge sources from Step 7)
2. An LLM (Azure OpenAI) for understanding queries and generating answers
3. Configuration for how to process queries and format responses

This knowledge base uses `ANSWER_SYNTHESIS` output mode, so the attached LLM generates a complete natural language answer from the retrieved chunks.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-search-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT.rstrip("/") + "/",
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Multi-source knowledge base over HR and health document indexes",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{KNOWLEDGE_BASE_NAME}' created or updated successfully.")


## Step 9: Query the multi-source knowledge base

Let's ask a two-part question that spans both knowledge sources:

* "What is the responsibility of the Zava CEO?" (HR-related)
* "What health plan would you recommend for the best mental health coverage?" (Health-related)

The knowledge base uses agentic retrieval to:

1. Analyze the query to understand two different topics
2. Decompose the query into focused subqueries
3. Determine which knowledge sources are relevant for each subquery
4. Run searches concurrently against the selected sources
5. Synthesize a unified answer from the merged results

In [ ]:
from azure.search.documents.knowledgebases.models import (
    SearchIndexKnowledgeSourceParams,
)

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

hrdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)
healthdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[
                KnowledgeBaseMessageTextContent(
                    text="What is the responsibility of the Zava CEO? "
                    "What health plan would you recommend if they wanted the best coverage for mental health services?"
                )
            ],
        )
    ],
    knowledge_source_params=[hrdocs_knowledge_source_params, healthdocs_knowledge_source_params],
    include_activity=True,
)

result = knowledge_base_client.retrieve(retrieval_request=retrieval_request)
display(Markdown(result.response[0].content[0].text))

activity_by_id = {activity.id: getattr(activity, "knowledge_source_name", None) for activity in (result.activity or [])}
references_json = []

for reference in result.references or []:
    reference_json = reference.as_dict()
    reference_json["knowledgeSourceName"] = activity_by_id.get(reference.activity_source, "unknown")
    references_json.append(reference_json)

print(json.dumps(references_json, indent=2))

## Bonus: Query from the Copilot CLI

Every knowledge base exposes an MCP server endpoint, which can be added to any MCP-compatible client like VS Code or GitHub Copilot CLI.
If you want to try out querying the knowledge base from the Copilot CLI, use the generated MCP configuration below and follow the steps in the [Copilot CLI sidequest](copilot-cli-sidequest.md).

In [ ]:
mcp_url = f"{AZURE_SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={AZURE_SEARCH_API_VERSION}"
headers = {"api-key": AZURE_SEARCH_ADMIN_KEY}
config = {"mcpServers": {KNOWLEDGE_BASE_NAME: {"type": "http", "url": mcp_url, "headers": headers}}}
print(json.dumps(config, indent=2))

## Summary

You've completed Part 1! You explored two types of knowledge sources, built knowledge bases with Azure OpenAI, and queried using agentic retrieval with citation-backed answers.

**Key concepts:**

* **File Knowledge Sources** accept raw uploads — no pre-built index required. The service handles chunking and embedding automatically.
* **Indexed Knowledge Sources** connect to pre-built search indexes for full control over schema and processing.
* A single knowledge base can reference multiple knowledge sources and intelligently routes queries to the relevant ones.
* Activity logs show how the agent decomposed and processed your query.

| | File Knowledge Source | Indexed Knowledge Source |
|---|---|---|
| Setup | Upload files directly | Pre-build a search index |
| Best for | Quick prototyping, small doc sets | Production workloads, large datasets |
| Control | Automatic chunking/embedding | Full control over schema and processing |

➡️ Continue to [Part 2: Add grounding from web results](part2-mai-grounding-mcp-kb.ipynb).